# Vegetation fraction (tree, shrub, grass) annual analysis

calculating annual area-weighted treeFrac, shrubFrac, and grassFrac over 60–90°N, and map Exp1/Exp4 GWL series. Spatial differences are also calculated for dynamic vegetation models ( in `frca_change.ipynb`).

In [28]:
from pathlib import Path
import sys

# Find this project from the notebook's working directory.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "utils" / "common.py").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Project folder containing utils/common.py was not found.")
    PROJECT_ROOT = PROJECT_ROOT.parent
CODE_DIR = PROJECT_ROOT / "utils"
sys.path.insert(0, str(CODE_DIR))

from common import find_project_root, load_ec_land_area, remap_land_area_to_esm, map_annual_to_gwl
PROJECT_ROOT = find_project_root(PROJECT_ROOT)
from frac import (
    annual_area_weighted_fraction,
    calculate_fraction_spatial_changes,
    describe_window,
    save_fraction_spatial_plot_data,
)
import xarray as xr

MODEL_CONFIGS = [
    {"name": "EC-Earth3-ESM-1", "family": "EC", "member": "r1i1p1f1"},
    {"name": "GISS-E2-1-G-CC2", "family": "NASA", "member": "r1i1p1f3"},
    {"name": "UKESM1-2-LL", "family": "UKESM", "member": "r1i1p1f1"},
]
EXPERIMENTS = ("Exp1", "Exp2", "Exp4")
GWL_LEGS = {"Exp1": "ramp-up", "Exp4": "ramp-down-2c"}
VARIABLES = ("treeFrac", "shrubFrac", "grassFrac")
LAT_MIN, LAT_MAX = 60.0, 90.0

# Keep these compact derived files outside the repository and raw-data folders.
OUTPUT_DIR = PROJECT_ROOT.parent / f"{PROJECT_ROOT.name}_plot_data" / "frac"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# Use each model's Exp1 treeFrac grid as the target for remapping EC land area. The area-weighting function selects 60–90°N.
ec_land_area = load_ec_land_area(PROJECT_ROOT)
land_areas = {}
for config in MODEL_CONFIGS:
    family = config["family"]
    sample_folder = PROJECT_ROOT / "Exp1_data" / family / f"Exp1_{family}_treeFrac"
    sample_file = sorted(sample_folder.glob("*.nc"))[0]
    with xr.open_dataset(sample_file) as sample:
        land_areas[config["name"]] = remap_land_area_to_esm(
            ec_land_area, sample["lat"].load().values, sample["lon"].load().values
        )

In [3]:
# Write one NetCDF per model. Annual values, GWL series, and means use separate dimensions so different experiment record lengths stay independent.
for config in MODEL_CONFIGS:
    model, family = config["name"], config["family"]
    annual = {experiment: {} for experiment in EXPERIMENTS}
    
    for experiment in EXPERIMENTS:
        base_folder = PROJECT_ROOT / f"{experiment}_data" / family
        for variable in VARIABLES:
            variable_folder = base_folder / f"{experiment}_{family}_{variable}"
            files = sorted(variable_folder.glob("*.nc"))
            annual[experiment][variable] = annual_area_weighted_fraction(
                files, variable, land_areas[model], lat_min=LAT_MIN, lat_max=LAT_MAX
            )
            series = annual[experiment][variable]
            print(f"{model} {experiment} {variable}: {series.sizes['year']} years; "
                  f"mean={float(series.mean('year')):.3f}%")

    mapped = {variable: {} for variable in VARIABLES}
    for variable in VARIABLES:
        for experiment, leg in GWL_LEGS.items():
            mapped[variable][experiment] = map_annual_to_gwl(
                annual[experiment][variable], model, config["member"], leg, PROJECT_ROOT
            )

    plot_data = xr.Dataset()
    for variable in VARIABLES:
        for experiment in EXPERIMENTS:
            series = annual[experiment][variable]
            plot_data[f"{variable}_{experiment.lower()}_annual"] = series.rename(
                {"year": f"{variable}_{experiment.lower()}_year"}
            )
        for experiment in GWL_LEGS:
            series = mapped[variable][experiment]
            plot_data[f"{variable}_{experiment.lower()}_gwl"] = series.rename(
                {"gwl": f"{variable}_{experiment.lower()}_gwl_level"}
            )
        plot_data[f"{variable}_exp2_full_record_mean"] = annual["Exp2"][variable].mean("year")

    plot_data.attrs.update({
        "model": model,
        "dataset_family": family,
        "latitude_band": "60–90°N",
        "units": "source units (usually %)",
        "description": "Regional annual vegetation fractions and GWL series; raw monthly grids are not included.",
    })
    filename = model.replace("/", "_").replace(" ", "_") + "_frac_plot_data.nc"
    output_file = OUTPUT_DIR / filename
    plot_data.to_netcdf(output_file)
    print(f"Saved plot data: {output_file}")


EC-Earth3-ESM-1 Exp1 treeFrac: 350 years; mean=21.193%
EC-Earth3-ESM-1 Exp1 shrubFrac: 350 years; mean=0.956%
EC-Earth3-ESM-1 Exp1 grassFrac: 350 years; mean=6.231%
EC-Earth3-ESM-1 Exp2 treeFrac: 500 years; mean=20.651%
EC-Earth3-ESM-1 Exp2 shrubFrac: 500 years; mean=0.641%
EC-Earth3-ESM-1 Exp2 grassFrac: 500 years; mean=5.254%
EC-Earth3-ESM-1 Exp4 treeFrac: 120 years; mean=12.106%
EC-Earth3-ESM-1 Exp4 shrubFrac: 120 years; mean=0.451%
EC-Earth3-ESM-1 Exp4 grassFrac: 120 years; mean=4.278%
Saved plot data: /mnt/e/lund/NGEA52/hysteresis/tipmip-hysteresis-highlat-ecosystem_plot_data/frac/EC-Earth3-ESM-1_frac_plot_data.nc
GISS-E2-1-G-CC2 Exp1 treeFrac: 290 years; mean=25.854%
GISS-E2-1-G-CC2 Exp1 shrubFrac: 290 years; mean=40.923%
GISS-E2-1-G-CC2 Exp1 grassFrac: 290 years; mean=8.235%
GISS-E2-1-G-CC2 Exp2 treeFrac: 300 years; mean=25.846%
GISS-E2-1-G-CC2 Exp2 shrubFrac: 300 years; mean=40.921%
GISS-E2-1-G-CC2 Exp2 grassFrac: 300 years; mean=8.235%
GISS-E2-1-G-CC2 Exp4 treeFrac: 97 years; 